### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>

### Features
Batch run folder structure + manifest generator.
- Each run = one scenario-parameter combination (sequential R-number)
- Each seed = one BehaviorSpace run of that combination.

### Prerequisites

### Usage 
- python make_runs_R125.py                # create folders + CSV in ./
- python make_runs_R125.py --root PATH    # repo root (default: current dir)
- python make_runs_R125.py --dry-run       # show what would be created

## Roadmap:

### Version 1


# imports

In [5]:
import os
import rasterio
import geopandas as gpd

import argparse
import csv
import hashlib
import os
from datetime import date
from pathlib import Path
import shutil

# init

In [11]:
from ti_city_00_helper_functions import *

case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
git_path = Path(r'C:\\GIT\\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\\GIT\\ti-city-model' or 'C:\\Users\\x\\GIT\\ti-city-model'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_model_path = Path(f'../model/{case_city}/')
ti_city_in_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2000,2025]
    simulation_years = [2025, 2050]

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# Defs

In [ ]:
def file_hash(path: Path) -> str:
    """Short hash of a file so the manifest pins the exact inputs used."""
    if not path.exists():
        return "n/a"
    return hashlib.md5(path.read_bytes()).hexdigest()[:8]


def create_folders(seeds_list, runs_list, first_run, run_type="--dry-run", root_folder_path = Path("./")) -> None:
    docs_folder_path = root_folder_path / "docs"
    outputs_folder_path = root_folder_path / f"model/{case_city}/data/out"
    inputs_folder_path = root_folder_path / f"model/{case_city}/data/in"

    if run_type == "--root":
        print("repo root (default: current dir)")
        help="repo root (default: current dir)"

    last_run = first_run + len(runs_list) - 1
    manifest_path = docs_folder_path / f"{case_city}_TA-model_manifest_R{first_run}_{last_run}.csv"
    if not manifest_path.exists():
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.touch()

    rows = []
    folders = []
    for run in runs_list:
        rnum = first_run + run["offset"]
        rname = f"R{rnum}"
        for seed in seeds_list:
            folder = outputs_folder_path / rname / f"seed{seed}"
            folders.append(folder)
            rows.append(
                {
                    "run": rname,
                    "scenario": run["scenario"],
                    "run_seed": seed,
                    "purpose": run["purpose"],
                    "parameter_variation": run["variation"],
                    "output_folder": str(folder.relative_to(outputs_folder_path)),
                    "code_version": code_version,
                    "inputs": inputs,
                    "mode_switches": model_mode,
                    "household_number": hh_number,
                    "scenarios_csv_md5": file_hash(inputs_folder_path / f"scenarios.csv"),
                    "weights_file_md5": file_hash(
                        inputs_folder_path / f"weights_{run['scenario']}.csv"
                    ),
                    "date_created": date.today().isoformat(),
                    "status": status,
                }
            )

    # ---- report plan ----
    print(f"Batch: {len(runs_list)} runs x {len(seeds_list)} seeds = {len(rows)} BehaviorSpace runs")
    for run in runs_list:
        print(f"  R{first_run + run['offset']:>3}  {run['scenario']:<12} {run['purpose']}")
    
    if run_type == "dry-run":
        action="store_true"
        for folder in folders:
            print(folder)
        print(f"\nManifest would be written to: {manifest_path}")
        print("\n(dry run — nothing created)")
        return
    elif run_type == "create":
    # ---- create folders ----
        for row in rows:
            folder = outputs_folder_path / row["run"] / f"seed{row['run_seed']}"
            folder.mkdir(parents=True, exist_ok=True)

            # Create "in" subfolder
            in_folder = folder / "in"
            in_folder.mkdir(parents=True, exist_ok=True)

            # copy .prj files to each folder
            prj_file_list = ['expansion_income_t000.prj', 'expansion_income_t005.prj',
                            'expansion_income_t010.prj', 'expansion_income_t015.prj',
                            'expansion_income_t020.prj', 'expansion_income_t025.prj']
            for prj_file in prj_file_list:
                prj_file_path = outputs_folder_path / f"{prj_file}"
                if prj_file_path.exists():
                    shutil.copy(prj_file_path, folder)
                else:
                    print(f"Warning: .prj file not found: {prj_file_path}")

            # Copy scenario and weights files to in folder
            scenario_file = inputs_folder_path / f"scenarios.csv"
            weights_file = inputs_folder_path / f"weights_{row['scenario']}.csv"

            if scenario_file.exists():
                shutil.copy(scenario_file, in_folder)
            else:
                print(f"Warning: scenario file not found: {scenario_file}")

            if weights_file.exists():
                shutil.copy(weights_file, in_folder)
            else:
                print(f"Warning: weights file not found: {weights_file}")

    print(f"\nCreated/verified {len(rows)} folders under {root_folder_path}")

    # ---- write manifest CSV ----
    with manifest_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote manifest: {manifest_path}")

In [ ]:
# ----------------------------------------------------------------------------
# Batch definition — R125+ seed-controlled calibration batch (v41 model,
# post-R108 code, corrected inputs: all-roads layer + year-2000 bu proximity)
# ----------------------------------------------------------------------------

seeds_list = [11, 22, 33]
first_run = 125

# (R-offset, scenario name in scenarios.csv, purpose, parameter group variation)
# Baseline weight files put weight 10 on the single associated parameter for
# EVERY agent class (dp, li, mi, hi, mi_red, hi_red), all other weights 0.
runs_list = [
    {
        "offset": 0,
        "scenario": "s0",
        "purpose": "Calibration scenario (expert weight set)",
        "variation": "Full mechanism set — expert-informed point weights, unchanged",
    },
    {
        "offset": 1,
        "scenario": "s_random",
        "purpose": "Null baseline: random allocation",
        "variation": "random weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 2,
        "scenario": "s_edge",
        "purpose": "Null baseline: edge growth",
        "variation": "bu weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 3,
        "scenario": "s_roadbound",
        "purpose": "Null baseline: road-bound growth",
        "variation": "road weight = 10 (all 6 agent rows), all other weights 0",
    },
]

code_version = "v41 (post-R108 allocation fixes)"
inputs = "all-roads layer + bu-proximity year 2000 (corrected vintage)"
model_mode = "calibrate? = TRUE, RED-on?, neighbour-on? on"
hh_number = "frozen (gross target ~29,559; per scenarios.csv)"
status = "pending"


create_folders(
    seeds_list, 
    runs_list, 
    first_run, run_type="dry-run", 
    root_folder_path = git_path,
    inputs=inputs,
    code_version=code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status
)

Batch: 4 runs x 3 seeds = 12 BehaviorSpace runs
  R125  s0           Calibration scenario (expert weight set)
  R126  s_random     Null baseline: random allocation
  R127  s_edge       Null baseline: edge growth
  R128  s_roadbound  Null baseline: road-bound growth
C:\GIT\ti-city-model\model\JAK\data\out\R125\seed11
C:\GIT\ti-city-model\model\JAK\data\out\R125\seed22
C:\GIT\ti-city-model\model\JAK\data\out\R125\seed33
C:\GIT\ti-city-model\model\JAK\data\out\R126\seed11
C:\GIT\ti-city-model\model\JAK\data\out\R126\seed22
C:\GIT\ti-city-model\model\JAK\data\out\R126\seed33
C:\GIT\ti-city-model\model\JAK\data\out\R127\seed11
C:\GIT\ti-city-model\model\JAK\data\out\R127\seed22
C:\GIT\ti-city-model\model\JAK\data\out\R127\seed33
C:\GIT\ti-city-model\model\JAK\data\out\R128\seed11
C:\GIT\ti-city-model\model\JAK\data\out\R128\seed22
C:\GIT\ti-city-model\model\JAK\data\out\R128\seed33

Manifest would be written to: C:\GIT\ti-city-model\docs\JAK_TA-model_manifest_R125_128.csv

(dry run — nothin

In [ ]:
code_version = "v41 (post-R108 allocation fixes)"
inputs = "all-roads layer + bu-proximity year 2000 (corrected vintage)"
model_mode = "calibrate? = TRUE, RED-on?, neighbour-on? on"
hh_number = "frozen (gross target ~29,559; per scenarios.csv)"
status = "pending"
runs_list = [
    {
        "offset": 0,
        "scenario": "s0",
        "purpose": "Calibration scenario (expert weight set)",
        "variation": "Full mechanism set — expert-informed point weights, unchanged",
    },
    {
        "offset": 1,
        "scenario": "s_random",
        "purpose": "Null baseline: random allocation",
        "variation": "random weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 2,
        "scenario": "s_edge",
        "purpose": "Null baseline: edge growth",
        "variation": "bu weight = 10 (all 6 agent rows), all other weights 0",
    },
    {
        "offset": 3,
        "scenario": "s_roadbound",
        "purpose": "Null baseline: road-bound growth",
        "variation": "road weight = 10 (all 6 agent rows), all other weights 0",
    },
]
create_folders(
    seeds_list, 
    runs_list, 
    first_run, run_type="create", 
    root_folder_path = git_path,
    inputs=inputs,
    code_version=code_version,
    model_mode=model_mode,
    hh_number=hh_number,
    status=status
)

Batch: 4 runs x 3 seeds = 12 BehaviorSpace runs
  R125  s0           Calibration scenario (expert weight set)
  R126  s_random     Null baseline: random allocation
  R127  s_edge       Null baseline: edge growth
  R128  s_roadbound  Null baseline: road-bound growth

Created/verified 12 folders under C:\GIT\ti-city-model
Wrote manifest: C:\GIT\ti-city-model\docs\JAK_TA-model_manifest_R125_128.csv


# The end (for now).